# Importando libs

In [21]:
import pandas as pd
import numpy as np
import os

In [22]:
def carregar_dados(caminho_entrada):
    print(f"Carregando dados de: {caminho_entrada}")
    try:
        df = pd.read_csv(caminho_entrada)
        print("Dataset carregado com sucesso!")
        return df
    except FileNotFoundError:
        print(f"Erro: O arquivo '{caminho_entrada}' não foi encontrado.")
        return None

In [23]:
def limpeza_basica(df):
    df_clean = df.copy()
    df_clean.columns = df_clean.columns.str.strip()
    print(f"Linhas após carregamento: {len(df_clean)}")

    df_clean.dropna(how='all', inplace=True)
    print(f"Linhas após remover linhas totalmente vazias: {len(df_clean)}")

    return df_clean


In [24]:
def tratar_colunas(df_clean):
    if 'release_date' in df_clean.columns:
        df_clean['release_date'] = pd.to_datetime(df_clean['release_date'], errors='coerce')
        print("- Coluna 'release_date' convertida para datetime.")

    for col in ['genres', 'categories', 'steamspy_tags']:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].fillna('').astype(str).str.split(';')
    print("- Colunas 'genres', 'categories', 'steamspy_tags' transformadas em listas.")

    return df_clean


In [25]:
def remover_duplicatas(df_clean):
    cols_lista = [col for col in ['genres', 'categories', 'steamspy_tags'] if col in df_clean.columns]
    for col in cols_lista:
        df_clean[col + '_str'] = df_clean[col].apply(lambda x: ';'.join(x) if isinstance(x, list) else '')

    duplicatas = df_clean.duplicated(subset=[col + '_str' for col in cols_lista]).sum()
    if duplicatas > 0:
        df_clean = df_clean.drop_duplicates(subset=[col + '_str' for col in cols_lista])
        print(f"- {duplicatas} linhas duplicadas foram removidas.")

    df_clean.drop(columns=[col + '_str' for col in cols_lista], inplace=True)
    print(f"Linhas após remover duplicatas: {len(df_clean)}")
    return df_clean


In [26]:
def tratar_plataformas(df_clean):
    if 'platforms' in df_clean.columns:
        df_clean['platform_windows'] = df_clean['platforms'].str.contains('windows', na=False)
        df_clean['platform_mac'] = df_clean['platforms'].str.contains('mac', na=False)
        df_clean['platform_linux'] = df_clean['platforms'].str.contains('linux', na=False)
        print("- Colunas de plataforma (windows, mac, linux) criadas.")

    colunas_irrelevantes = [col for col in ['url'] if col in df_clean.columns]
    if colunas_irrelevantes:
        df_clean.drop(columns=colunas_irrelevantes, inplace=True)

    return df_clean


In [27]:
def engenharia_features(df_clean):
    """Cria features essenciais para os modelos de ML."""
    print("Iniciando engenharia de features...")
    
    if 'price' in df_clean.columns:
        df_clean['is_free'] = (df_clean['price'] == 0).astype(int)
        print("– Feature 'is_free' criada.")
    
    if 'categories' in df_clean.columns:
        df_clean['is_multiplayer'] = df_clean['categories'].str.contains('Multi-player', na=False).astype(int)
        print("– Feature 'is_multiplayer' criada.")
        
    if 'positive_ratings' in df_clean.columns and 'negative_ratings' in df_clean.columns:
        df_clean['total_ratings'] = df_clean['positive_ratings'] + df_clean['negative_ratings']
        print("– Feature 'total_ratings' criada.")
        
    if 'average_playtime' in df_clean.columns:
        df_clean['average_playtime_hours'] = df_clean['average_playtime'] / 60
        print("– Feature 'average_playtime_hours' criada.")
        
    # Remover colunas irrelevantes
    colunas_irrelevantes = [col for col in ['url'] if col in df_clean.columns]
    if colunas_irrelevantes:
        df_clean.drop(columns=colunas_irrelevantes, inplace=True)
        print(f"– Colunas irrelevantes ({colunas_irrelevantes}) removidas.")
        
    return df_clean

In [28]:
def salvar_dados(df_clean, caminho_saida):
    diretorio_saida = os.path.dirname(caminho_saida)
    if diretorio_saida and not os.path.exists(diretorio_saida):
        os.makedirs(diretorio_saida)
    df_clean.to_csv(caminho_saida, index=False, encoding='utf-8')
    print(f"Processo concluído! Arquivo limpo salvo em: {caminho_saida}")


In [29]:
caminho_dados_steam = "../data/raw/steam.csv"
caminho_dados_tags = "../data/raw/steamspy_tag_data.csv"
caminho_dados_limpos = "../data/processed/steam_cleaned.csv"

df_steam = carregar_dados(caminho_dados_steam)
df_tags = carregar_dados(caminho_dados_tags)

if df_steam is not None and df_tags is not None:
    
    df_steam.rename(columns={'appid': 'app_id'}, inplace=True)
    df_tags.rename(columns={'appid': 'app_id'}, inplace=True)
    

    print("\nIniciando pipeline de limpeza e features...")
    df_merged = pd.merge(df_steam, df_tags, on='app_id', how='left')
    print(f"Dados mesclados: {df_merged.shape}")
    
    # 4. Pipeline de Limpeza e Features
    df_limpo = limpeza_basica(df_merged)
    df_limpo = engenharia_features(df_limpo) # -> NOVO PASSO (essencial)
    df_limpo = tratar_colunas(df_limpo)
    df_limpo = remover_duplicatas(df_limpo)
    df_limpo = tratar_plataformas(df_limpo)
    
    # 5. Salvar
    salvar_dados(df_limpo, caminho_dados_limpos)
    
    # 6. Verificar Resultado
    print("\nAmostra do DataFrame final:")
    display(df_limpo.head())

    print("\nInformações do DataFrame final:")
    df_limpo.info()
else:
    print("Processamento interrompido pois um dos arquivos não foi carregado.")

Carregando dados de: ../data/raw/steam.csv
Dataset carregado com sucesso!
Carregando dados de: ../data/raw/steamspy_tag_data.csv
Dataset carregado com sucesso!

Iniciando pipeline de limpeza e features...
Dados mesclados: (27075, 389)
Linhas após carregamento: 27075
Linhas após remover linhas totalmente vazias: 27075
Iniciando engenharia de features...
– Feature 'is_free' criada.
– Feature 'is_multiplayer' criada.
– Feature 'total_ratings' criada.
– Feature 'average_playtime_hours' criada.
- Coluna 'release_date' convertida para datetime.
- Colunas 'genres', 'categories', 'steamspy_tags' transformadas em listas.
- 7704 linhas duplicadas foram removidas.
Linhas após remover duplicatas: 19371
- Colunas de plataforma (windows, mac, linux) criadas.


/var/folders/pk/8wd2jncs1fq_p68kh1_6fglc0000gn/T/ipykernel_29358/2068625544.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean.drop(columns=[col + '_str' for col in cols_lista], inplace=True)


Processo concluído! Arquivo limpo salvo em: ../data/processed/steam_cleaned.csv

Amostra do DataFrame final:


,app_id,name,release_date,english,developer,publisher,platforms,required_age,categories,genres,...,wrestling,zombies,e_sports,is_free,is_multiplayer,total_ratings,average_playtime_hours,platform_windows,platform_mac,platform_linux
0,10,Counter-Strike,2000-11-01,1,Valve,Valve,windows;mac;linux,0,"[Multi-player, Online Multi-Player, Local Mult...",[Action],...,0,0,550,0,1,127873,293.533333,True,True,True
2,30,Day of Defeat,2003-05-01,1,Valve,Valve,windows;mac;linux,0,"[Multi-player, Valve Anti-Cheat enabled]",[Action],...,0,0,0,0,1,3814,3.116667,True,True,True
4,50,Half-Life: Opposing Force,1999-11-01,1,Gearbox Software,Valve,windows;mac;linux,0,"[Single-player, Multi-player, Valve Anti-Cheat...",[Action],...,0,0,0,0,1,5538,10.400000,True,True,True
5,60,Ricochet,2000-11-01,1,Valve,Valve,windows;mac;linux,0,"[Multi-player, Online Multi-Player, Valve Anti...",[Action],...,0,0,0,0,1,3442,2.916667,True,True,True
6,70,Half-Life,1998-11-08,1,Valve,Valve,windows;mac;linux,0,"[Single-player, Multi-player, Online Multi-Pla...",[Action],...,0,0,0,0,1,28855,21.666667,True,True,True



Informações do DataFrame final:
<class 'pandas.core.frame.DataFrame'>
Index: 19371 entries, 0 to 27072
Columns: 396 entries, app_id to platform_linux
dtypes: bool(3), datetime64[ns](1), float64(2), int64(382), object(8)
memory usage: 58.3+ MB


In [30]:
if df_limpo is not None:
    print("\nAmostra do DataFrame final:")
    display(df_limpo.head())

    print("\nInformações do DataFrame final:")
    df_limpo.info()



Amostra do DataFrame final:


,app_id,name,release_date,english,developer,publisher,platforms,required_age,categories,genres,...,wrestling,zombies,e_sports,is_free,is_multiplayer,total_ratings,average_playtime_hours,platform_windows,platform_mac,platform_linux
0,10,Counter-Strike,2000-11-01,1,Valve,Valve,windows;mac;linux,0,"[Multi-player, Online Multi-Player, Local Mult...",[Action],...,0,0,550,0,1,127873,293.533333,True,True,True
2,30,Day of Defeat,2003-05-01,1,Valve,Valve,windows;mac;linux,0,"[Multi-player, Valve Anti-Cheat enabled]",[Action],...,0,0,0,0,1,3814,3.116667,True,True,True
4,50,Half-Life: Opposing Force,1999-11-01,1,Gearbox Software,Valve,windows;mac;linux,0,"[Single-player, Multi-player, Valve Anti-Cheat...",[Action],...,0,0,0,0,1,5538,10.400000,True,True,True
5,60,Ricochet,2000-11-01,1,Valve,Valve,windows;mac;linux,0,"[Multi-player, Online Multi-Player, Valve Anti...",[Action],...,0,0,0,0,1,3442,2.916667,True,True,True
6,70,Half-Life,1998-11-08,1,Valve,Valve,windows;mac;linux,0,"[Single-player, Multi-player, Online Multi-Pla...",[Action],...,0,0,0,0,1,28855,21.666667,True,True,True



Informações do DataFrame final:
<class 'pandas.core.frame.DataFrame'>
Index: 19371 entries, 0 to 27072
Columns: 396 entries, app_id to platform_linux
dtypes: bool(3), datetime64[ns](1), float64(2), int64(382), object(8)
memory usage: 58.3+ MB
